# Split + Augmentation — YouTube Toxic Comments
**Proyecto NLP | Detección de Mensajes de Odio**  
**Rama:** `feature/split_model_v03`

---

## 🎯 Objetivo
1. Dividir el dataset en train/test (80/20) **antes** de cualquier augmentation
2. Aplicar back-translation moderada **solo sobre train**
3. Exportar los splits listos para el modelado

## ⚠️ Regla de oro
> **Primero split → luego augmentation. Nunca al revés.**  
> Si aumentamos antes del split, datos sintéticos del mismo comentario  
> podrían estar en train Y en test → data leakage → métricas infladas.

## 1. 📦 Importaciones

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from deep_translator import GoogleTranslator
import random
import time
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

# Ruta raíz robusta — funciona desde cualquier subcarpeta
BASE_DIR = Path().resolve()
while not (BASE_DIR / 'pyproject.toml').exists():
    BASE_DIR = BASE_DIR.parent

IN_PATH        = BASE_DIR / 'data' / 'processed' / 'toxic_comments_clean_v03.csv'
TRAIN_PATH     = BASE_DIR / 'data' / 'processed' / 'toxic_train_v03.csv'
TRAIN_AUG_PATH = BASE_DIR / 'data' / 'processed' / 'toxic_train_augmented_v03.csv'
TEST_PATH      = BASE_DIR / 'data' / 'processed' / 'toxic_test_v03.csv'

print('✅ Librerías cargadas')
print(f'📂 Input:      {IN_PATH.name}')
print(f'💾 Train:      {TRAIN_PATH.name}')
print(f'💾 Train aug:  {TRAIN_AUG_PATH.name}')
print(f'💾 Test:       {TEST_PATH.name}')

✅ Librerías cargadas
📂 Input:      toxic_comments_clean_v03.csv
💾 Train:      toxic_train_v03.csv
💾 Train aug:  toxic_train_augmented_v03.csv
💾 Test:       toxic_test_v03.csv


## 2. 📂 Carga del dataset preprocesado

In [4]:
df = pd.read_csv(IN_PATH)

print(f'📐 Shape: {df.shape}')
print(f'📋 Columnas: {list(df.columns)}')
print(f'\n=== ⚖️ Distribución IsToxic ===')
counts = df['IsToxic'].value_counts()
pcts   = df['IsToxic'].value_counts(normalize=True) * 100
for val in [True, False]:
    label = '🔴 Tóxico   ' if val else '🟢 No tóxico'
    print(f'  {label}: {counts[val]:>4} ({pcts[val]:.1f}%)')
df.head(3)

📐 Shape: (997, 5)
📋 Columnas: ['CommentId', 'VideoId', 'Text_original', 'Text_clean', 'IsToxic']

=== ⚖️ Distribución IsToxic ===
  🔴 Tóxico   :  459 (46.0%)
  🟢 No tóxico:  538 (54.0%)


,CommentId,VideoId,Text_original,Text_clean,IsToxic
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,people would take step back make case anyone e...,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,law enforcement train shoot apprehend train sh...,True
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\r\nDont you reckon them 'black lives matter' ...,reckon black life matter banner hold white cun...,True


## 3. Train / Test Split (80/20)

Usamos `stratify=y` para mantener la misma proporción de clases en train y test.

In [5]:
X = df['Text_clean']
y = df['IsToxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # mantiene proporción 46/54 en ambos splits
)

print('=== ✂️ Resultado del Split ===')
print(f'  Train: {len(X_train):>4} muestras ({len(X_train)/len(df)*100:.1f}%)')
print(f'  Test:  {len(X_test):>4} muestras ({len(X_test)/len(df)*100:.1f}%)')

print(f'\n=== ⚖️ Distribución en TRAIN ===')
train_counts = y_train.value_counts()
for val in [True, False]:
    label = '🔴 Tóxico   ' if val else '🟢 No tóxico'
    pct = train_counts[val] / len(y_train) * 100
    print(f'  {label}: {train_counts[val]:>4} ({pct:.1f}%)')

print(f'\n=== ⚖️ Distribución en TEST ===')
test_counts = y_test.value_counts()
for val in [True, False]:
    label = '🔴 Tóxico   ' if val else '🟢 No tóxico'
    pct = test_counts[val] / len(y_test) * 100
    print(f'  {label}: {test_counts[val]:>4} ({pct:.1f}%)')

# Guardar DataFrames de train y test base
df_train = pd.DataFrame({'Text_clean': X_train, 'IsToxic': y_train}).reset_index(drop=True)
df_test  = pd.DataFrame({'Text_clean': X_test,  'IsToxic': y_test}).reset_index(drop=True)

df_test.to_csv(TEST_PATH, index=False)
df_train.to_csv(TRAIN_PATH, index=False)
print(f'\n✅ Test guardado:  {TEST_PATH.name}')
print(f'✅ Train guardado: {TRAIN_PATH.name}')

=== ✂️ Resultado del Split ===
  Train:  797 muestras (79.9%)
  Test:   200 muestras (20.1%)

=== ⚖️ Distribución en TRAIN ===
  🔴 Tóxico   :  367 (46.0%)
  🟢 No tóxico:  430 (54.0%)

=== ⚖️ Distribución en TEST ===
  🔴 Tóxico   :   92 (46.0%)
  🟢 No tóxico:  108 (54.0%)

✅ Test guardado:  toxic_test_v03.csv
✅ Train guardado: toxic_train_v03.csv


## 4. 🌍 Back-Translation — Augmentation moderada

### ¿Por qué back-translation y no WordNet?
- WordNet puede cambiar `black → dark` → desastre semántico en hate speech
- Back-translation preserva el significado global del comentario
- Genera variedad léxica real sin inventar odio

### Estrategia:
- Solo aumentamos comentarios **tóxicos** del train
- Solo el **25%** de los tóxicos (moderado, no masivo)
- Idiomas intermedios: **español** y **francés**
- El test **nunca se toca**

In [6]:
def back_translate(text: str, mid_lang: str = 'es') -> str:
    """
    Traduce texto al idioma intermedio y de vuelta al inglés.
    Genera una paráfrasis semánticamente equivalente.
    """
    try:
        # Inglés → idioma intermedio
        translated = GoogleTranslator(
            source='en', target=mid_lang
        ).translate(text)

        # Idioma intermedio → inglés
        back = GoogleTranslator(
            source=mid_lang, target='en'
        ).translate(translated)

        return back if back else text
    except Exception:
        return text  # si falla la API, devuelve el original


# --- Test rápido ---
print('=== 🔍 Test back-translation ===')
ejemplos = [
    ("you stupid idiot go back where came", 'es'),
    ("hate black people police wrong shoot", 'fr'),
]
for texto, lang in ejemplos:
    resultado = back_translate(texto, lang)
    print(f'\nOriginal ({lang}): {texto}')
    print(f'Back-translated:  {resultado}')

=== 🔍 Test back-translation ===



Original (es): you stupid idiot go back where came
Back-translated:  Stupid idiot, go back the way you came.

Original (fr): hate black people police wrong shoot
Back-translated:  hate black people, the police shoot badly


In [ ]:
# --- Aplicar augmentation al 25% de los tóxicos del train ---
toxic_train = df_train[df_train['IsToxic'] == True].copy()
n_to_augment = int(len(toxic_train) * 0.25)

print(f'📊 Tóxicos en train:     {len(toxic_train)}')
print(f'📊 A aumentar (25%):     {n_to_augment}')
print(f'⏳ Aplicando back-translation... (puede tardar unos minutos)')

# Selección aleatoria del 25%
sample_idx = random.sample(list(toxic_train.index), n_to_augment)
to_augment = toxic_train.loc[sample_idx]

augmented_rows = []
langs = ['es', 'fr']  # alternamos español y francés

for i, (idx, row) in enumerate(to_augment.iterrows()):
    lang = langs[i % 2]  # alternar idiomas
    aug_text = back_translate(row['Text_clean'], lang)

    # Solo añadimos si el texto cambió (si no cambió no aporta variedad)
    if aug_text != row['Text_clean'] and len(aug_text.strip()) > 0:
        augmented_rows.append({
            'Text_clean': aug_text,
            'IsToxic': True
        })

    # Pequeña pausa para no saturar la API
    time.sleep(0.3)

    if (i + 1) % 10 == 0:
        print(f'  Procesados: {i+1}/{n_to_augment}')

df_augmented = pd.DataFrame(augmented_rows)
print(f'\n✅ Comentarios aumentados generados: {len(df_augmented)}')

📊 Tóxicos en train:     367
📊 A aumentar (25%):     91
⏳ Aplicando back-translation... (puede tardar unos minutos)
  Procesados: 10/91
  Procesados: 20/91
  Procesados: 30/91
  Procesados: 40/91


In [ ]:
# --- Combinar train original + aumentados ---
df_train_aug = pd.concat([df_train, df_augmented], ignore_index=True)

print('=== 📊 Train ANTES de augmentation ===')
bc = df_train['IsToxic'].value_counts()
for val in [True, False]:
    print(f'  {"Tóxico" if val else "No tóxico"}: {bc[val]} ({bc[val]/len(df_train)*100:.1f}%)')

print('\n=== 📊 Train DESPUÉS de augmentation ===')
ac = df_train_aug['IsToxic'].value_counts()
for val in [True, False]:
    print(f'  {"Tóxico" if val else "No tóxico"}: {ac[val]} ({ac[val]/len(df_train_aug)*100:.1f}%)')

print(f'\n📐 Shape train original:  {df_train.shape}')
print(f'📐 Shape train aumentado: {df_train_aug.shape}')

# Guardar train aumentado
df_train_aug.to_csv(TRAIN_AUG_PATH, index=False)
print(f'\n✅ Train aumentado guardado: {TRAIN_AUG_PATH.name}')

=== 📊 Train ANTES de augmentation ===
  Tóxico: 367 (46.0%)
  No tóxico: 430 (54.0%)

=== 📊 Train DESPUÉS de augmentation ===
  Tóxico: 454 (51.4%)
  No tóxico: 430 (48.6%)

📐 Shape train original:  (797, 2)
📐 Shape train aumentado: (884, 2)

✅ Train aumentado guardado: toxic_train_augmented_v03.csv


## 5. 🔍 Verificación de ejemplos aumentados

In [ ]:
# Mostrar algunos ejemplos de back-translation
print('=== 🔍 Ejemplos de comentarios aumentados ===')
sample_aug = df_augmented.sample(min(5, len(df_augmented)), random_state=42)

for i, (_, row) in enumerate(sample_aug.iterrows()):
    print(f'\n[{i+1}] Back-translated: {row["Text_clean"][:120]}')

=== 🔍 Ejemplos de comentarios aumentados ===

[1] Back-translated: couple pop shot air clock disperse like cockroach light spin

[2] Back-translated: cnn nonsense

[3] Back-translated: I guess sucking construction worker shows the whole video wrong.

[4] Back-translated: stefan molyneux hates black women very surprised damn brits conquer absolutely nothing freedom win decade go hate pale o

[5] Back-translated: dumbest people American land


## 6. 📋 Resumen del Split y Augmentation

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════╗
║        RESUMEN SPLIT + AUGMENTATION V03                      ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  INPUT: toxic_comments_clean_v03.csv ({len(df)} filas){'':>18}║
║                                                              ║
║  SPLIT 80/20 con stratify                                    ║
║  • Train base: {len(df_train):>4} muestras                            ║
║  • Test:       {len(df_test):>4} muestras (NUNCA se toca)             ║
║                                                              ║
║  AUGMENTATION (solo en train)                                ║
║  • Técnica: Back-translation (ES + FR)                       ║
║  • Solo tóxicos, 25% del total tóxico                        ║
║  • Comentarios nuevos generados: {len(df_augmented):>4}                   ║
║  • Train aumentado: {len(df_train_aug):>4} muestras                      ║
║                                                              ║
║  ARCHIVOS GENERADOS                                          ║
║  • toxic_train_v03.csv          (baseline)                   ║
║  • toxic_train_augmented_v03.csv (con augmentation)          ║
║  • toxic_test_v03.csv            (test — no tocar)           ║
║                                                              ║
║  SIGUIENTE PASO                                              ║
║  → Model_Training_v03.ipynb                                  ║
║     Fase 1: TF-IDF + Logistic Regression                      ║
║             SIN augmentation → métricas + MLflow            ║
║                                                              ║
║    Fase 2: TF-IDF + Logistic Regression                      ║
║             CON augmentation → métricas + MLflow            ║
║                                                              ║
║    Fase 3: TF-IDF + LightGBM                                 ║
║             SIN augmentation → métricas + MLflow            ║
║                                                              ║
║    Fase 4: TF-IDF + LightGBM                                 ║
║             CON augmentation → métricas + MLflow            ║
║                                                              ║
║    Fase 5: Tabla comparativa final                           ║
║             Accuracy | F1 | Recall | ROC-AUC                                  ║
╚══════════════════════════════════════════════════════════════╝
""")

NameError: name 'df' is not defined